# Emotion Detection from Text using Bidirectional LSTM

* Developed a deep learning model to classify **6 human emotions** from text using NLP techniques.

* Implemented a **Bidirectional LSTM** architecture with word embeddings to capture contextual meaning of sentences.

* Achieved high classification accuracy on the **Emotions Dataset** (joy, sadness, anger, fear, surprise, disgust).

* Visualized results using **Confusion Matrix** and **Classification Report**.

---
**Dataset:** [Emotions Dataset for NLP – Kaggle](https://www.kaggle.com/datasets/praveengovi/emotions-dataset-for-nlp)

**Download and place files:** `train.txt`, `val.txt`, `test.txt` in your working directory or Google Drive.

# Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import os

from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout, SpatialDropout1D
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.utils import to_categorical

import warnings
warnings.filterwarnings('ignore')

print("All libraries imported successfully!")

# Configuration & Hyperparameters

In [ ]:
# --- Paths ---
TRAIN_PATH = "train.txt"
VAL_PATH   = "val.txt"
TEST_PATH  = "test.txt"

# --- NLP Parameters ---
MAX_VOCAB   = 20000    # Maximum vocabulary size
MAX_LEN     = 50       # Max tokens per sentence (padding/truncation)
EMBED_DIM   = 128      # Embedding vector size

# --- Training Parameters ---
BATCH_SIZE  = 64
EPOCHS      = 30
NUM_CLASS   = 6

print("Configuration set!")

# Load Dataset

In [ ]:
def load_data(path):
    """
    Loads the emotions dataset.
    Each line format: 'sentence;label'
    """
    texts, labels = [], []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if ';' in line:
                text, label = line.rsplit(';', 1)
                texts.append(text.strip())
                labels.append(label.strip())
    return texts, labels

train_texts, train_labels = load_data(TRAIN_PATH)
val_texts,   val_labels   = load_data(VAL_PATH)
test_texts,  test_labels  = load_data(TEST_PATH)

print(f"Train samples  : {len(train_texts)}")
print(f"Val samples    : {len(val_texts)}")
print(f"Test samples   : {len(test_texts)}")
print(f"\nSample text  : {train_texts[0]}")
print(f"Sample label : {train_labels[0]}")

# Exploratory Data Analysis (EDA)

In [ ]:
# Class distribution
all_labels = train_labels + val_labels + test_labels
label_counts = pd.Series(all_labels).value_counts()

plt.figure(figsize=(8, 4))
sns.barplot(x=label_counts.index, y=label_counts.values, palette='viridis')
plt.title('Emotion Class Distribution')
plt.xlabel('Emotion')
plt.ylabel('Count')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

print("\nClass distribution:")
print(label_counts)

In [ ]:
# Sentence length distribution
lengths = [len(t.split()) for t in train_texts]

plt.figure(figsize=(8, 4))
plt.hist(lengths, bins=40, color='steelblue', edgecolor='black')
plt.title('Sentence Length Distribution (Training Set)')
plt.xlabel('Number of Words')
plt.ylabel('Frequency')
plt.axvline(MAX_LEN, color='red', linestyle='--', label=f'MAX_LEN = {MAX_LEN}')
plt.legend()
plt.tight_layout()
plt.show()

print(f"Max sentence length  : {max(lengths)}")
print(f"Mean sentence length : {np.mean(lengths):.1f}")
print(f"95th percentile      : {np.percentile(lengths, 95):.0f}")

# Text Preprocessing

In [ ]:
def clean_text(text):
    """Basic text cleaning."""
    text = text.lower()
    text = re.sub(r"http\S+", "", text)          # remove URLs
    text = re.sub(r"[^a-z\s]", "", text)          # keep only letters & spaces
    text = re.sub(r"\s+", " ", text).strip()      # normalize whitespace
    return text

train_texts_clean = [clean_text(t) for t in train_texts]
val_texts_clean   = [clean_text(t) for t in val_texts]
test_texts_clean  = [clean_text(t) for t in test_texts]

print("Before:", train_texts[0])
print("After :", train_texts_clean[0])

# Tokenization & Padding

In [ ]:
# Build tokenizer on training data ONLY (no data leakage)
tokenizer = Tokenizer(num_words=MAX_VOCAB, oov_token='<OOV>')
tokenizer.fit_on_texts(train_texts_clean)

vocab_size = min(MAX_VOCAB, len(tokenizer.word_index) + 1)
print(f"Vocabulary size: {vocab_size}")

# Convert texts to sequences
X_train = pad_sequences(tokenizer.texts_to_sequences(train_texts_clean), maxlen=MAX_LEN, padding='post', truncating='post')
X_val   = pad_sequences(tokenizer.texts_to_sequences(val_texts_clean),   maxlen=MAX_LEN, padding='post', truncating='post')
X_test  = pad_sequences(tokenizer.texts_to_sequences(test_texts_clean),  maxlen=MAX_LEN, padding='post', truncating='post')

print(f"X_train shape: {X_train.shape}")
print(f"X_val shape  : {X_val.shape}")
print(f"X_test shape : {X_test.shape}")

# Label Encoding

In [ ]:
le = LabelEncoder()
le.fit(train_labels)

class_names = list(le.classes_)
print("Classes:", class_names)

y_train = to_categorical(le.transform(train_labels), NUM_CLASS)
y_val   = to_categorical(le.transform(val_labels),   NUM_CLASS)
y_test  = to_categorical(le.transform(test_labels),  NUM_CLASS)

print(f"y_train shape: {y_train.shape}")
print(f"y_test shape : {y_test.shape}")

# Build Bidirectional LSTM Model

In [ ]:
model = Sequential([
    # Word Embedding Layer
    Embedding(input_dim=vocab_size, output_dim=EMBED_DIM, input_length=MAX_LEN),
    SpatialDropout1D(0.2),

    # First BiLSTM Layer
    Bidirectional(LSTM(128, return_sequences=True)),
    Dropout(0.3),

    # Second BiLSTM Layer
    Bidirectional(LSTM(64, return_sequences=False)),
    Dropout(0.3),

    # Dense Layers
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(NUM_CLASS, activation='softmax')
])

model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

# Callbacks

In [ ]:
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    ModelCheckpoint(
        'best_emotion_bilstm.keras',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    )
]

print("Callbacks configured!")

# Train Model

In [ ]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1
)

# Plot Training Curves

In [ ]:
# Accuracy
plt.figure(figsize=(8, 4))
plt.plot(history.history['accuracy'],     label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.title('Training vs Validation Accuracy')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Loss
plt.figure(figsize=(8, 4))
plt.plot(history.history['loss'],     label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Training vs Validation Loss')
plt.legend()
plt.tight_layout()
plt.show()

# Evaluate Model

In [ ]:
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Loss     : {test_loss:.4f}")
print(f"Test Accuracy : {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")

# Predictions

In [ ]:
y_pred_proba = model.predict(X_test)
y_pred       = np.argmax(y_pred_proba, axis=1)
y_true       = np.argmax(y_test, axis=1)

print(f"Predictions generated for {len(y_pred)} samples.")

# Classification Report

In [ ]:
print(classification_report(
    y_true,
    y_pred,
    target_names=class_names
))

# Confusion Matrix

In [ ]:
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=class_names,
    yticklabels=class_names
)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix - Emotion Detection')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

# Predict on Custom Text Samples

In [ ]:
def predict_emotion(text, model, tokenizer, le, max_len=MAX_LEN):
    """
    Predict the emotion for a single input sentence.
    """
    cleaned = clean_text(text)
    seq     = tokenizer.texts_to_sequences([cleaned])
    padded  = pad_sequences(seq, maxlen=max_len, padding='post', truncating='post')
    proba   = model.predict(padded, verbose=0)[0]
    idx     = np.argmax(proba)
    emotion = le.classes_[idx]
    confidence = proba[idx] * 100
    return emotion, confidence, proba


# --- Test on custom sentences ---
samples = [
    "I am so happy today, everything feels wonderful!",
    "I feel really sad and alone right now.",
    "This makes me so angry, I can't believe it happened.",
    "I'm terrified about what might happen next.",
    "Wow, I did not expect that at all, what a surprise!",
    "That was absolutely disgusting, I feel sick."
]

print(f"{'Sentence':<55} {'Predicted Emotion':<15} {'Confidence':>10}")
print("-" * 85)
for s in samples:
    emotion, conf, _ = predict_emotion(s, model, tokenizer, le)
    display = s[:52] + '...' if len(s) > 55 else s
    print(f"{display:<55} {emotion:<15} {conf:>9.1f}%")

# Probability Distribution for a Single Sample

In [ ]:
sample_text = "I am so happy and excited about this news!"
emotion, confidence, proba = predict_emotion(sample_text, model, tokenizer, le)

plt.figure(figsize=(7, 4))
colors = ['green' if i == np.argmax(proba) else 'steelblue' for i in range(NUM_CLASS)]
plt.bar(le.classes_, proba * 100, color=colors, edgecolor='black')
plt.xlabel('Emotion')
plt.ylabel('Probability (%)')
plt.title(f'Emotion Probabilities\n"{sample_text}"')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()

print(f"Predicted: {emotion} ({confidence:.1f}% confidence)")